[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/toche7/SlideAdvanceDSBDI/blob/main/notebook/notebook-12-13-llm-data-science.ipynb)

**Open this notebook in Google Colab**

ถ้ากด badge ด้านบน จะเปิด notebook นี้จาก GitHub เข้า Colab ได้ทันที

# Notebook Demo: LLM with Data Science
**BDAT501 Data Science Fundamentals — Module 12-13 (Afternoon)**

ใน notebook นี้ เราจะสาธิตการใช้ LLM ในขั้นตอนต่าง ๆ ของ Data Science workflow:
- Data Extraction (Sentiment analysis, Entity recognition)
- Data Validation (Anomaly detection)
- Data Generation (Synthetic data)
- Data Summarization (Business narrative)
- Chatbot (Simple Q&A interface)

เราจะใช้ **GROQ API** ซึ่งให้ API access ฟรีกับ LLM ที่เร็วและมีความสามารถดี

## 1. Setup and Installation

In [ ]:
# ติดตั้ง library ที่จำเป็น
%pip install groq -q
%pip install python-dotenv -q

import json
import pandas as pd
from groq import Groq

# ใช้ model เดียวกันทั้ง notebook เพื่อเปลี่ยนครั้งเดียวจบ
MODEL_NAME = "openai/gpt-oss-20b"

print("Setup complete")

## 2. GROQ API Setup

ได้ API key ฟรีได้จาก: https://console.groq.com/keys

**ขั้นตอน:**
1. สมัครสมาชิก GROQ
2. ไปที่ API Keys section
3. สร้าง API key ใหม่
4. วาง key ที่นี่

In [ ]:
# ใส่ GROQ API key โดยอ่านจาก Colab Secret เป็นหลัก
import os

GROQ_API_KEY = None

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = None

if not GROQ_API_KEY:
    GROQ_API_KEY = os.getenv("GROQ_API_KEY", "").strip()

if not GROQ_API_KEY:
    from getpass import getpass
    GROQ_API_KEY = getpass("Enter GROQ API key: ").strip()

# สร้าง Groq client
client = Groq(api_key=GROQ_API_KEY)

# ทดลอง connection
try:
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": "Say 'Hello from GROQ'"}],
        model=MODEL_NAME,
        max_tokens=50
    )
    print("✓ GROQ API connected successfully")
    print(f"Model: {MODEL_NAME}")
    print(f"Response: {response.choices[0].message.content}")
except Exception as e:
    print(f"❌ Error: {e}")
    print("Please check your Colab Secret, environment variable, or model name")

Simple wrapping เพื่อเรียนใช้ chat completion 

In [ ]:
def chat_completion(prompt, model=MODEL_NAME, max_tokens=1000):
    """Simple wrapper for client.chat.completions.create, automatically setting the role to 'user'."""
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        messages=messages,
        model=model,
        max_tokens=max_tokens
    )

    result_text = response.choices[0].message.content.strip()
    # Handle cases where the response might be wrapped in markdown code blocks
    if result_text.startswith("```json"):
        result_text = result_text.replace("```json", "").replace("```", "")
    elif result_text.startswith("```"):
        result_text = result_text.replace("```", "")

    return result_text

In [ ]:
chat_completion('หาดใหญ่เป็นจังหวัดในภาคไหน')

## 3. Sentiment Analysis

ให้ LLM วิเคราะห์ sentiment ของ feedback จากลูกค้า

In [ ]:
# ข้อมูล feedback ตัวอย่าง
feedback_list = [
    "แอปช้ามากและมี bug เรื่อง login ที่ทำให้ผมไม่สามารถเข้าบัญชีได้นาน",
    "ใช้งานได้ดี UI สวยมาก ขอบคุณที่ใช้ font ไทยที่ชัด",
    "ปัญหากับการโอนเงินบ่อยครั้ง ทำให้สูญเสียเงินไปแล้ว",
    "ระบบ notification ทำให้หงุดหงิด เพราะบอกข้อมูลไม่ถูกต้องบ่อย",
    "ยอดเยี่ยม! ทีมสนับสนุนลูกค้าตอบเร็วมากและช่วยแก้ปัญหาได้"
]

def extract_sentiment(feedback_text):
    """ใช้ GROQ LLM ดึง sentiment และ urgency จากข้อความ"""

    prompt = f"""Analyze the following feedback in Thai and extract:
1. sentiment (positive, negative, neutral)
2. urgency (low, medium, high)
3. summary (brief 1-line summary)

Return as JSON format ONLY, no extra text:
{{
  "sentiment": "...",
  "urgency": "...",
  "summary": "..."
}}

Feedback: {feedback_text}
"""
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model=MODEL_NAME,
        max_tokens=1000
    )

    result_text = response.choices[0].message.content.strip()
    if result_text.startswith("```json"):
        result_text = result_text.replace("```json", "").replace("```", "")
    elif result_text.startswith("```"):
        result_text = result_text.replace("```", "")

    return json.loads(result_text.strip())

print("Processing sentiment analysis...\n")
sentiment_results = []
for fb in feedback_list:
    try:
        result = extract_sentiment(fb)
        result["original_feedback"] = fb
        sentiment_results.append(result)
        print(f"✓ {fb[:50]}...")
    except Exception as e:
        print(f"✗ Error: {e}")

df_sentiment = pd.DataFrame(sentiment_results)
df_sentiment[["sentiment", "urgency", "summary"]]

In [ ]:
df_sentiment

## 4. Entity Recognition 

ดึง entities/ประเด็นสำคัญจาก feedback

In [ ]:
def extract_entities(feedback_text):
    """ใช้ GROQ LLM ดึง entities/issues จากข้อความ"""

    prompt = f"""Analyze the following feedback in Thai and extract:
1. main entities/issues (list 2-4 main topics)
2. impact level (low, medium, high)

Return as JSON format ONLY, no extra text:
{{
  "entities": [...],
  "impact": "..."
}}

Feedback: {feedback_text}
"""
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model=MODEL_NAME,
        max_tokens=1000
    )

    result_text = response.choices[0].message.content.strip()
    if result_text.startswith("```json"):
        result_text = result_text.replace("```json", "").replace("```", "")
    elif result_text.startswith("```"):
        result_text = result_text.replace("```", "")

    return json.loads(result_text.strip())

print("Processing entity recognition...\n")
entity_results = []
for fb in feedback_list:
    try:
        result = extract_entities(fb)
        result["original_feedback"] = fb
        entity_results.append(result)
        print(f"✓ {fb[:50]}...")
    except Exception as e:
        print(f"✗ Error: {e}")

df_entities = pd.DataFrame(entity_results)
df_entities[["entities", "impact"]]

## 5. Anomaly Detection
ตรวจสอบความผิดปกติของข้อมูล เช่น outlier หรือ missing value

In [ ]:
# ข้อมูลตัวอย่าง บางอันผิดปกติ
customer_records = [
    {"customer_id": "C001", "age": 35, "tenure": 24, "total_charges": 2500},
    {"customer_id": "C002", "age": 250, "tenure": 60, "total_charges": 5000},  # อายุผิด
    {"customer_id": "C003", "age": 28, "tenure": -5, "total_charges": 1200},  # tenure ติดลบ
    {"customer_id": "C004", "age": 45, "tenure": 12, "total_charges": -800},  # ค่าใช้จ่ายติดลบ
    {"customer_id": "C005", "age": 32, "tenure": 48, "total_charges": 15000},  # ค่าใช้จ่ายสูงผิดปกติ
]

def detect_anomalies(records):
    """ให้ LLM วิเคราะห์ anomalies ในข้อมูลลูกค้า"""
    data_str = json.dumps(records, indent=2)
    prompt = f"""Analyze the following customer records for anomalies:
1. Identify records with obvious errors (negative values, unrealistic ages, etc.)
2. Flag records that seem statistically unusual
3. For each anomaly, suggest what the issue might be

Return as JSON:
{{
  "anomalies": [
    {{
      "customer_id": "...",
      "issue_type": "error|unusual",
      "fields_affected": [...],
      "recommendation": "..."
    }}
  ],
  "summary": "..."
}}

Data:
{data_str}
"""

    response = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model=MODEL_NAME,
        max_tokens=2000
    )

    result_text = response.choices[0].message.content.strip()
    if result_text.startswith("```json"):
        result_text = result_text.replace("```json", "").replace("```", "")
    elif result_text.startswith("```"):
        result_text = result_text.replace("```", "")

    return json.loads(result_text.strip())

print("Detecting anomalies...\n")
anomaly_report = detect_anomalies(customer_records)

print(f"Summary: {anomaly_report['summary']}\n")
print("Issues found:")
for anomaly in anomaly_report["anomalies"]:
    print(f"  - {anomaly['customer_id']}: {anomaly['recommendation']}")

## 6. Data Generation: Synthetic Customer Feedback

ให้ LLM สร้างข้อมูล feedback สำหรับ demo หรือ testing

In [ ]:
def generate_synthetic_feedback(issue_category, n=3):
    """ให้ LLM สร้าง synthetic feedback สำหรับหมวดหมู่ที่กำหนด"""

    prompt = f"""Generate {n} realistic customer feedback in Thai language for the following issue category: {issue_category}

Requirements:
- Make feedback varied (some positive, some negative, some mixed)
- Include specific details and examples
- Keep each feedback 1-2 sentences
- Write in natural Thai language

Return as JSON array of strings ONLY:
[
  "feedback 1",
  "feedback 2",
  "feedback 3"
]
"""

    response = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model=MODEL_NAME,
        max_tokens=2000
    )

    result_text = response.choices[0].message.content.strip()
    if result_text.startswith("```json"):
        result_text = result_text.replace("```json", "").replace("```", "")
    elif result_text.startswith("```"):
        result_text = result_text.replace("```", "")

    return json.loads(result_text.strip())

print("Generating synthetic feedback...\n")

categories = ["Payment Issues", "App Speed", "Customer Support"]
synthetic_data = {}

for category in categories:
    feedback = generate_synthetic_feedback(category, n=2)
    synthetic_data[category] = feedback
    print(f"\n📝 {category}:")
    for i, fb in enumerate(feedback, 1):
        print(f"  {i}. {fb}")

## 7. Data Summarization: Business Narrative

ให้ LLM แปลผลการวิเคราะห์เป็นรายงาน business

In [ ]:
def generate_business_summary(analysis_results, target_audience="Executive"):
    """ให้ LLM สรุป analysis ผลให้เป็นภาษาธุรกิจ"""

    data_str = json.dumps(analysis_results, indent=2, ensure_ascii=False)

    prompt = f"""Based on the following customer feedback analysis, write a business summary in Thai.

Target Audience: {target_audience}

Requirements:
- 3-4 paragraphs
- Highlight key insights and recommendations
- Use business-friendly language
- Focus on actionable items

Analysis Data:
{data_str}
"""

    response = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model=MODEL_NAME,
        max_tokens=2000
    )

    return response.choices[0].message.content

print("Generating business summary...\n")

if "sentiment" in df_sentiment.columns and "urgency" in df_sentiment.columns and len(df_sentiment) > 0:
    summary_data = {
        "total_feedback": len(df_sentiment),
        "sentiment_distribution": df_sentiment["sentiment"].value_counts().to_dict(),
        "high_urgency_count": len(df_sentiment[df_sentiment["urgency"] == "high"]),
        "top_issues": ["App Performance", "Payment System", "UI/UX"]
    }

    summary = generate_business_summary(summary_data, target_audience="Product Manager")
    print("EXECUTIVE SUMMARY\n")
    print(summary)
else:
    print("No sentiment/urgency data found. Run cells 7 and 9 first and ensure JSON parsing succeeds.")

## 8. Key Takeaways

### ✅ ที่ LLM ช่วยได้จริง:
1. **Extraction** — ดึง sentiment, entities จากข้อความจำนวนมาก
2. **Validation** — ช่วยตรวจสอบ anomaly ที่อาจหลุด
3. **Generation** — สร้าง synthetic data สำหรับ demo
4. **Summarization** — แปลผลเชิงเทคนิคเป็นภาษาธุรกิจ

### ⚠️ ต้องระวัง:
- ทุกผลลัพธ์ต้องตรวจโดยมนุษย์ก่อนใช้
- ยังไม่ควรอิงตัดสินใจ production โดยพูดตรง ๆ ว่า LLM บอกแล้ว

### 🚀 ต่อยอด:
- Combine with RAG (Retrieval-Augmented Generation) สำหรับคำตอบที่อิงข้อมูลจริง
- Use vector databases (Pinecone, Milvus) สำหรับ semantic search
- Deploy chatbot เป็น web service เพื่อให้ทีมใช้จริง